In [1]:
from openai import OpenAI, AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool

In [2]:
from agents.tracing import set_tracing_disabled

set_tracing_disabled(True)

local_model = OpenAIChatCompletionsModel(
    model="gemma4:31b-cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )
)

In [4]:
agent = Agent(
    name="History Tutor",
    instructions="You answer history questions clearly and concisely.",
    model=local_model,
    # tools=[get_weather]
)
result = await Runner.run(agent, "What is the capital of Pakistan?")
print(result.final_output)

The capital of Pakistan is **Islamabad**.


In [5]:
import nest_asyncio
nest_asyncio.apply()

In [6]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled
import asyncio

# Disable tracing warning
set_tracing_disabled(True)

# Step 1: Connect to Ollama
client = AsyncOpenAI(
    api_key="ollama",  # dummy key
    base_url="http://127.0.0.1:11434/v1"
)

# Step 2: Define local model
local_model = OpenAIChatCompletionsModel(
    model="gemma4:31b-cloud",  # make sure this exists
    openai_client=client
)

# Step 3: Attach model to agent
agent = Agent(
    name="Analyst",
    instructions="You analyze topics briefly.",
    model=local_model   # 🔥 IMPORTANT
)

# Step 4: Run
async def main():
    result = await Runner.run(agent, "Why is Python popular for AI?")

    print("=" * 50)
    print(f"Final Output:  {result.final_output[:100]}...")
    print(f"Last Agent:    {result.last_agent.name}")
    print(f"New Items:     {len(result.new_items)}")
    print(f"Raw Responses: {len(result.raw_responses)}")
    print("=" * 50)

asyncio.run(main())

Final Output:  Python is the dominant language for AI primarily due to the following factors:

**1. Extensive Ecosy...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


In [7]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled

# Disable tracing warnings
set_tracing_disabled(True)

# Step 1: Connect to Ollama local server
client = AsyncOpenAI(
    api_key="ollama",  # dummy key for local server
    base_url="http://127.0.0.1:11434/v1"
)

# Step 2: Define the model
local_model = OpenAIChatCompletionsModel(
    model="gemma4:31b-cloud",  # Use "gemma4:31b-cloud" if you want cloud
    openai_client=client
)

# Step 3: Create the agent with model
agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model  # 🔥 model is now properly attached
)

# --- Multi-turn conversation ---
# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: A **list** is a built-in Python data structure used to store a collection of items in a single variable.

Here are the key characteristics:

1.  **Ordered:** They maintain the order in which items are inserted.
2.  **Mutable:** You can change, add, or remove items after the list is created.
3.  **Dynamic:** They can hold items of different data types (integers, strings, or even other lists).
4.  **Indexed:** Each item has a position starting at index `0`.

### Basic Example:
```python
# Creating a list
fruits = ["apple", "banana", "cherry"]

# Accessing by index
print(fruits[0])  # Output: apple

# Modifying an item
fruits[1] = "blueberry"

# Adding an item
fruits.append("orange")

print(fruits) # Output: ['apple', 'blueberry', 'cherry', 'orange']
```

Turn 2: There are two primary ways to sort a list, depending on whether you want to modify the original list or create a new one.

### 1. The `.sort()` Method (In-place)
Use this to permanently change the order of the original li

In [9]:
import asyncio
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
from agents.run import RunConfig

# Ollama client (OpenAI-compatible)
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Required but unused by Ollama
)

# Model pointing to your Ollama instance
ollama_model = OpenAIChatCompletionsModel(
    model="gemma4:31b-cloud",
    openai_client=ollama_client,
)

# Run config
run_config = RunConfig(model=ollama_model)

agent = Agent(name="Streamer", instructions="Explain briefly.")

async def main():
    # Streaming — see tokens as they arrive
    result = Runner.run_streamed(
        agent,
        "What is machine learning in 2 sentences?",
        run_config=run_config,
    )

    async for event in result.stream_events():
        if event.type == "raw_response_event" and hasattr(event.data, "delta"):
            delta = event.data.delta
            # Handle both string delta and object with text attribute
            if isinstance(delta, str):
                print(delta, end="", flush=True)
            elif hasattr(delta, "text"):
                print(delta.text, end="", flush=True)

    print(f"\n\n--- Final: {result.final_output}")

asyncio.run(main())

Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. It uses algorithms to improve their performance over time as they are exposed to more information.

--- Final: Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. It uses algorithms to improve their performance over time as they are exposed to more information.
